In [0]:
from scripts.utils import silver_utils
print(dir(silver_utils))

In [0]:
from pyspark.sql.functions import (
    col, when, sum, lit, trim, lower, upper,
    to_date, current_timestamp,round
)
from datetime import datetime
import uuid,logging

In [0]:
df=spark.read.table("workspace.ecommerce_bronze.df_products")
logging.info('table read successfully')

In [0]:
silver_utils.check_schema(df)

In [0]:
df=silver_utils.cast_types(df,{
'product_id' : "string",
 'product_category_name': "string",
 'product_weight_g': "double",
 'product_length_cm' : "double",
 'product_height_cm': "double",
 'product_width_cm' : "double"})

In [0]:
silver_utils.null_profiling(df,"df_products")
df=silver_utils.handle_nulls_drop(df,drop_cols=["product_id","product_category_name"])

In [0]:
df=silver_utils.handle_duplicates(df,["product_id"])

In [0]:
df=silver_utils.standardize_strings(df,{
    "product_id" : lambda c : trim(c),
    "product_category_name" : lambda c : lower(trim(c))
})

In [0]:
checks = [
    #nulls
    (
        "product_id",
        "product_id not null",
        col("product_id").isNotNull()
    ),
    (
        "product_category_name",
        "product_category_name not null",
        col("product_category_name").isNotNull()
    ),

    #business rules
    (
        "product_weight_g",
        "reasonable weight",
        (col("product_weight_g") > 0) & (col("product_weight_g") < 100000)
    ),
    (
        "product_length_cm",
        "reasonable length",
        (col("product_length_cm") > 0) & (col("product_length_cm") < 500)
    ),
    (
        "product_height_cm",
        "reasonable height",
        (col("product_height_cm") > 0) & (col("product_height_cm") < 500)
    ),
    (
        "product_width_cm",
        "reasonable width",
        (col("product_width_cm") > 0) & (col("product_width_cm") < 500)
    )
]

dq_table = silver_utils.build_dq_table(
    spark=spark,
    df=df,
    checks=checks,
    table_name="df_products",
    warn_threshold=5.0
)

dq_table.write.mode("overwrite").saveAsTable("workspace.ecommerce_silver.df_products_dq")
dq_saved = spark.read.table("workspace.ecommerce_silver.df_products_dq")
display(dq_saved)

In [0]:
df=df.withColumn("product_volume_cm3",col("product_length_cm")*col("product_height_cm")*col("product_width_cm"))
df=df.withColumn('density_g_per_cm3',round(col('product_weight_g')/col('product_volume_cm3'),2))

In [0]:
df=silver_utils.add_silver_metadata(df)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.ecommerce_silver.df_products")
print("saved successfully !")

In [0]:
%sql
--sanity check
SELECT * FROM workspace.ecommerce_silver.df_products LIMIT 10